# Spike del modelo de lenguaje · INV-30

Compara los modelos candidatos para el agente de InventAI/o con el conjunto v0 (43 preguntas: 20 base y 23 variantes del usuario). El diseño está en `docs/INV-30-requerimientos.md` §4.4 y en la §5.1 de `docs/SPRINT-6-requerimientos.md`.

**Dónde corre**
- **En el equipo de desarrollo** (VS Code, kernel con el entorno `~/venvs/inventaio`): Ollama nativo en WSL (`bash scripts/ollama_wsl.sh iniciar`) con `qwen3:8b`, `qwen3:4b-instruct-2507` y `llama3.1:8b`. **Esta es la medición oficial** (la demostración del Release 2 corre aquí).
- **En Colab** (Entorno de ejecución → Cambiar tipo → **L4**): el cuaderno instala Ollama 0.40.5, clona el repositorio y baja los modelos, incluidos `qwen3:14b` y `qwen3:30b-a3b-instruct-2507` como techo de calidad (no caben en 8 GB).

**Regla de los datos (N9, D22).** En Colab corre la variante `colab` del conjunto: sucursales con nombres ficticios (NORTE, CENTRO, SUR), otra fecha y cantidades sintéticas. Nunca se usa la bodega real fuera del equipo. Ningún modelo corre en servidores externos (`OLLAMA_NO_CLOUD=1`).

**Partes:** 1) preparar el entorno · 2) modelos · 3) explorar preguntas, una o varias a la vez y con varios modelos lado a lado · 4) examen completo y reporte · 5) rendimiento · 6) en Colab, descargar los resultados.

## 1. Entorno

In [ ]:
import os, sys, subprocess, time, json, asyncio, argparse, glob
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
RAMA = "feature/INV-30-nlp-agent"        # develop, después del merge del spike
OLLAMA_VERSION = "0.40.5"                # la misma del equipo de desarrollo

if EN_COLAB:
    REPO = "/content/InventaIO"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,
                        "https://github.com/ddelgadillod/InventaIO.git", REPO], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx", "pydantic-settings"], check=True)
    if not os.path.exists("/usr/local/bin/ollama"):
        subprocess.run(f"curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh", shell=True, check=True)
    entorno_ollama = {**os.environ, "OLLAMA_NO_CLOUD": "1", "OLLAMA_NUM_PARALLEL": "1",
                      "OLLAMA_MAX_LOADED_MODELS": "1", "OLLAMA_KEEP_ALIVE": "30m"}
    subprocess.Popen(["ollama", "serve"], env=entorno_ollama, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(5)
    VARIANTE = "colab"
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    ENTORNO = "colab-" + (gpu.split()[-1].lower() if gpu else "cpu")
else:
    REPO = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    VARIANTE, ENTORNO = "local", "local"

sys.path.insert(0, os.path.join(REPO, "nlp_agent"))
BASE_URL = os.environ.get("LLM_BASE_URL", "http://localhost:11434")

import httpx
print("Ollama", httpx.get(BASE_URL + "/api/version", timeout=10).json()["version"], "en", BASE_URL)
print("Variante del conjunto:", VARIANTE, "· carpeta de resultados:", ENTORNO)

## 2. Modelos

En Colab se descargan los que falten. En el equipo no se descarga nada: el disco es limitado y ya están los tres del spike.

In [ ]:
MODELOS_EQUIPO = ["qwen3:8b", "qwen3:4b-instruct-2507-q4_K_M", "llama3.1:8b"]
MODELOS_TECHO = ["qwen3:14b", "qwen3:30b-a3b-instruct-2507-q4_K_M"]     # solo con 24 GB de VRAM (L4)
QWEN_CON_RAZONAMIENTO = {"qwen3:8b", "qwen3:14b"}                      # W4: los demás no razonan

MODELOS = MODELOS_EQUIPO + (MODELOS_TECHO if EN_COLAB else [])
instalados = {m["name"] for m in httpx.get(BASE_URL + "/api/tags", timeout=10).json()["models"]}
for m in MODELOS:
    if m not in instalados and EN_COLAB:
        print("Descargando", m, "…")
        subprocess.run(["ollama", "pull", m], check=True)
instalados = {m["name"]: m for m in httpx.get(BASE_URL + "/api/tags", timeout=10).json()["models"]}
DISPONIBLES = [m for m in MODELOS if m in instalados]
for m in MODELOS:
    estado = f"digest {instalados[m]['digest'][:12]}, {instalados[m]['size'] / 1e9:.1f} GB" if m in instalados else "no está"
    print(f"{m:38} {estado}")

## 3. Explorar preguntas

Elige una o varias preguntas del conjunto (con Ctrl o Mayús) o escribe la tuya, uno o varios modelos, y pulsa **Preguntar**. Para cada combinación verás:
- las herramientas que llamó el modelo, ronda por ronda;
- la respuesta final;
- la calificación (solo en las preguntas del conjunto);
- el tiempo y los tokens por ronda.

Las herramientas responden con los resultados sintéticos de cada pregunta. En una pregunta escrita por ti devuelven listas vacías, así que sirve para ver qué herramienta elige el modelo, no la respuesta.

In [ ]:
import html
import ipywidgets as w
from IPython.display import display, HTML

from core.config import Settings
from evaluacion.arnes import correr_pregunta
from evaluacion.conjunto import Pregunta, cargar, herramientas
from llm.cliente import ClienteOllama

conjunto = cargar(VARIANTE)
catalogo = herramientas(VARIANTE)
sucursal_admin = "NORTE" if VARIANTE == "colab" else "LA 21"

sel_preguntas = w.SelectMultiple(options=[(f"{p.id} · {p.pregunta}", p.id) for p in conjunto.preguntas],
                                 value=(conjunto.preguntas[0].id,), rows=10, description="Preguntas", layout=w.Layout(width="98%"))
libre = w.Text(placeholder="…o escribe otra pregunta (reemplaza la selección)", description="Libre", layout=w.Layout(width="98%"))
rol = w.Dropdown(options=[("Gerente", "gerente"), ("Adm. Bodega Central", "admin_bodega"),
                          (f"Adm. sucursal {sucursal_admin}", "admin_sucursal")], description="Rol (libre)")
sel_modelos = w.SelectMultiple(options=DISPONIBLES, value=tuple(DISPONIBLES[:1]), rows=len(DISPONIBLES), description="Modelos")
modo = w.ToggleButtons(options=[("Con GPU", "gpu"), ("Sin GPU", "cpu")], description="Modo")
razonamiento = w.Checkbox(description="Razonamiento (Qwen3 8B y 14B)")
boton = w.Button(description="Preguntar", button_style="primary", icon="play")
salida = w.Output()

def _celda(c, pregunta_del_conjunto):
    llamadas = "<br>".join(f"{x['ronda']}. <code>{x['herramienta']}</code> {html.escape(json.dumps(x['argumentos'], ensure_ascii=False))}"
                           for x in c.llamadas) or "<i>ninguna</i>"
    respuesta = html.escape(c.texto).replace(chr(10), "<br>") or f"<b>error:</b> {c.error}"
    cal = ""
    if pregunta_del_conjunto:
        k = c.calificacion
        marcas = " ".join(f"{'✅' if k[x] else '❌'} {x}" for x in ("herramienta", "fidelidad", "hechos", "prohibido", "aclaracion", "rechazo", "sin_error"))
        cal = f"<p><b>{'ACIERTO' if k['acierto'] else 'FALLA'}</b> · {marcas}" + (
            f"<br>cifras sin respaldo: {k['cifras_sin_respaldo']}" if k["cifras_sin_respaldo"] else "") + "</p>"
    return (f"<td style='vertical-align:top;padding:6px'><p><b>Herramientas</b><br>{llamadas}</p>"
            f"<p><b>Respuesta</b><br>{respuesta}</p>{cal}"
            f"<p style='color:gray'>{c.ms_total / 1000:.1f} s · {c.rondas} rondas · {c.tokens_prompt_max} tokens por ronda</p></td>")

def mostrar(contenido):
    # append_display_data y no `with salida:`: con código asíncrono, ipywidgets pierde
    # la captura después de un await y la salida se va a otra celda
    salida.append_display_data(HTML(contenido))

async def preguntar():
        salida.outputs = ()          # clear_output() se aplica tarde y borraría lo primero que se agrega
        if libre.value.strip():
            preguntas = [Pregunta(id="libre", rol=rol.value, sucursal_usuario={"gerente": None, "admin_bodega": "BODEGA_CENTRAL",
                         "admin_sucursal": sucursal_admin}[rol.value], tipo="abstencion", pregunta=libre.value.strip(), abstencion=True)]
            del_conjunto = False
        else:
            preguntas = [conjunto.por_id(i) for i in sel_preguntas.value]
            del_conjunto = True
        for p in preguntas:
            mostrar(f"<h4>{p.id} · {html.escape(p.pregunta)} <small>({p.rol}{', ' + p.sucursal_usuario if p.sucursal_usuario else ''})</small></h4>")
            if del_conjunto:
                esperado = " o ".join(f"<code>{e.herramienta}</code> {html.escape(json.dumps(e.argumentos, ensure_ascii=False))}" for e in p.esperado) or "ninguna herramienta"
                mostrar(f"<p style='color:gray'>Se espera: {esperado}" + (f" · nota: {html.escape(p.nota)}" if p.nota else "") + "</p>")
            columnas, encabezados = [], []
            for m in sel_modelos.value:
                ajustes = Settings(OLLAMA_BASE_URL=BASE_URL, LLM_MODELO=m, LLM_TIMEOUT_S=300,
                                   LLM_RAZONAMIENTO=razonamiento.value and m in QWEN_CON_RAZONAMIENTO)
                cliente = ClienteOllama(ajustes)
                c = await correr_pregunta(cliente, p, catalogo, conjunto.fecha_datos,
                                          opciones={"num_gpu": 0} if modo.value == "cpu" else None)
                await cliente.cerrar()
                encabezados.append(f"<th style='text-align:left'>{m}</th>")
                columnas.append(_celda(c, del_conjunto))
            mostrar(f"<table style='width:100%;border-collapse:collapse'><tr>{''.join(encabezados)}</tr><tr>{''.join(columnas)}</tr></table>")

boton.on_click(lambda _: asyncio.ensure_future(preguntar()))
display(sel_preguntas, libre, rol, sel_modelos, modo, razonamiento, boton, salida)

## 4. Examen completo y reporte

Corre las 43 preguntas × 3 corridas en cada modelo y modo, y guarda una línea por corrida en `nlp_agent/evaluacion/resultados/<fecha>_<entorno>/`. Con GPU son unos minutos por modelo. **Sin GPU** (`"cpu"`) puede tardar horas por modelo: conviene dejarlo corriendo de noche, o empezar con `CORRIDAS = 1`.

El reporte da, por modelo:
- el acierto y **pass^3** (las 3 corridas aciertan);
- la herramienta correcta, la fidelidad de cifras, la abstención y las fugas;
- el p95 del tiempo y los tokens por ronda.

También compara los modelos de a pares con la diferencia pareada por pregunta y su intervalo del 95 %. Si el intervalo cruza 0, los modelos no se separan, y se decide por tamaño y latencia.

In [ ]:
from evaluacion import correr as correr_mod
from evaluacion import reporte
from IPython.display import Markdown

EXAMEN_MODELOS = DISPONIBLES        # ajusta la lista si quieres correr solo algunos
MODOS = ["gpu"]                     # agrega "cpu" para medir sin GPU (criterio 9; lento)
CORRIDAS = 3

archivos = []
for m in EXAMEN_MODELOS:
    for modo_examen in MODOS:
        print(f"=== {m} · {modo_examen}")
        args = argparse.Namespace(modelo=m, modo=modo_examen, corridas=CORRIDAS, preguntas=None, variante=VARIANTE,
                                  entorno=ENTORNO, base_url=BASE_URL, timeout=600.0 if modo_examen == "cpu" else 180.0,
                                  num_ctx=8192, razonamiento=False, salida=None)
        archivos.append(await correr_mod.correr(args))
display(Markdown(reporte.markdown(reporte.leer(archivos))))

In [ ]:
# El reporte de todo lo que hay en la carpeta de este entorno (también de corridas anteriores)
todos = sorted(glob.glob(os.path.join(REPO, "nlp_agent/evaluacion/resultados", f"*_{ENTORNO}", "*.jsonl")))
display(Markdown(reporte.markdown(reporte.leer([Path(x) for x in todos]))))

## 5. Rendimiento (criterio 3)

Mide, para cada modelo:
- la carga en frío;
- la latencia al primer token;
- la velocidad de lectura del prompt (sin contar lo que Ollama reutiliza);
- la generación;
- la memoria.

Lo hace sobre la respuesta final que ve el usuario. La última línea mide la latencia de Qwen3 con razonamiento (W4).

In [ ]:
from evaluacion import smoke

for m in EXAMEN_MODELOS:
    for modo_smoke in MODOS:
        smoke.correr(argparse.Namespace(modelo=m, modo=modo_smoke, razonamiento=False, variante=VARIANTE,
                                        entorno=ENTORNO, base_url=BASE_URL, salida=None))
if "qwen3:8b" in EXAMEN_MODELOS:
    smoke.correr(argparse.Namespace(modelo="qwen3:8b", modo="gpu", razonamiento=True, variante=VARIANTE,
                                    entorno=ENTORNO, base_url=BASE_URL, salida=None))

## 6. Colab: descargar los resultados

Descarga un zip con la carpeta de resultados de este entorno para versionarla en el repositorio (N9: los resultados se guardan en el repositorio).

In [ ]:
if EN_COLAB:
    import shutil
    from google.colab import files
    carpeta = os.path.join(REPO, "nlp_agent/evaluacion/resultados")
    shutil.make_archive(f"/content/resultados_spike_{ENTORNO}", "zip", carpeta)
    files.download(f"/content/resultados_spike_{ENTORNO}.zip")
else:
    print("En el equipo, los resultados ya están en nlp_agent/evaluacion/resultados/")